# Random-features mass model -- testing lengthscale "blocks"

This notebook tests `CHIMERA.population.mass.paired.random_features_density`
(the fixed random-features / random-Fourier-features paired mass model) with
its 4 `feature_scales` (the mixture of lengthscales spanning broad-body to
narrow-peak structure) treated as **sampled hyperparameters** with **narrow
priors** around the defaults, alongside `w_out`. It mirrors
`examples/bpl3p_run_tempest.py` (same priors-dict / `tempest` / hyperlikelihood
structure) but swaps in the random-features mass model.

**Section 0** is fully local (no cluster data, no `tempest` needed) and
sanity-checks the plumbing: prior-predictive draws of different lengthscale
"blocks", capacity diagnostics, and a jit/vmap check that sampling
`feature_scales` doesn't trigger per-particle recompilation.

**Section 1** mirrors `bpl3p_run_tempest.py` for an actual `tempest`
hyperlikelihood run -- it needs the GWTC5 PE/injection files and the
`tempest` package (cluster-only), so edit `dir_data`/`dir_chain` before
running it.

## A necessary fix that shipped alongside this notebook

`feature_scales` used to be an `eqx.field(static=True)` tuple baked directly
into the fixed random draw (`W_hidden ~ N(0, 1/feature_scales[i]**2)` drawn
once). That makes it physically impossible to *sample*: a JAX `static` field
has to be identical, by value, across calls for the jit cache to reuse a
compiled trace, so a sampler proposing a new `feature_scales` on every
particle/step would force a fresh XLA compilation per particle -- a
non-starter for `tempest`'s vectorized particle population.

`CHIMERA/population/mass/paired/nn.py` now draws a fixed, feature_scales-
independent standard-normal noise tensor `raw_W_hidden` once, and applies the
lengthscale as an ordinary differentiable division at call time:
`W_hidden[k] = raw_W_hidden[k] / feature_scales[block_of(k)]`. This makes
`feature_scales` a genuine dynamic pytree leaf -- sampled, differentiated,
and `vmap`'d exactly like `w_out` already was, with a single jit trace
regardless of the concrete values drawn. See the updated class docstring in
`nn.py` for the full reasoning.

In [ ]:
import os, sys
os.environ["CHIMERA_ENABLE_GPU"] = "True"
sys.path.append(os.getcwd() + '/../')

import numpy as np
import scipy.stats
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from CHIMERA.population.mass.paired import random_features_density
from CHIMERA.population.mass import p_m1, p_m1m2

## Config

`default_feature_scales` are the 4 lengthscales from the class docstring
(broad-body -> narrow-peak). `scale_prior_width` sets how narrow the priors
around them are (fractional half-width in log-space); at 0.2 the 4 resulting
bands (`[0.8,1.2]`, `[0.24,0.36]`, `[0.08,0.12]`, `[0.024,0.036]`) are both
"quite narrow" and non-overlapping, which matters: if the bands overlapped,
the posterior would pick up a spurious label-switching degeneracy between
blocks (block *i* and block *i+1* trading places), since nothing else ties a
specific block to a specific physical role.

In [ ]:
n_features = 64                                    # keep modest (tens, not hundreds) -- see nn.py docstring
default_feature_scales = (1.0, 0.3, 0.1, 0.03)     # the 4 lengthscale "blocks"
n_scales = len(default_feature_scales)
scale_prior_width = 0.2                            # +/-20% narrow, non-overlapping log-uniform bands
w_out_prior_scale = 1.5                            # std of the Normal prior on each w_out entry

master_key = jax.random.PRNGKey(0)

# tempest / hyperlikelihood run settings (Section 1) -- modest defaults for a *test* run;
# bump these up for a production fit, same knobs as bpl3p_run_tempest.py's argparse flags.
n_particles = 256
n_steps = 5
n_total = 4096
prng_key = 0
resample = 'syst'

dir_data = '/leonardo_work/IscrC_MLGW/gwtc5/data/'
dir_chain = '/leonardo_work/IscrC_MLGW/gwtc5/res/tempest/'
chain_label = 'gwtc5_randomfeatures_blocks_tempest'

## Priors

Same style as `bpl3p_run_tempest.py`: scipy.stats-based `uniform`/`loguniform`/`norm` helpers, one prior per scalar hyperparameter. `w_out{i}` (one per feature) and `scale{i}` (one per block, 1-indexed) are later re-assembled into the `w_out`/`feature_scales` arrays that `random_features_density.update()` expects.

In [ ]:
def uniform(low, high):
  return scipy.stats.uniform(loc=low, scale=high - low)

def loguniform(low, high):
  return scipy.stats.loguniform(a=low, b=high)

def norm(loc, scale):
  return scipy.stats.norm(loc=loc, scale=scale)

# --- mass-model (random_features_density) priors ---
mass_priors = {
  "beta": uniform(-4., 12.),               # mass-ratio pairing power law
  "m_low": uniform(0.4, 1.4),
  "m_high": uniform(50., 200.),
}

# narrow, non-overlapping priors on the 4 lengthscale blocks
scale_priors = {
  f"scale{i+1}": loguniform(s * (1. - scale_prior_width), s * (1. + scale_prior_width))
  for i, s in enumerate(default_feature_scales)
}

# one Normal prior per random-features output weight
w_out_priors = {f"w_out{i}": norm(loc=0., scale=w_out_prior_scale) for i in range(n_features)}

# --- cosmology / rate priors (same as bpl3p_run_tempest.py) ---
cosmo_rate_priors = {
  "H0": uniform(10., 200.),
  "gamma": uniform(0., 12.),
  "kappa": uniform(0., 6.),
  "zp": uniform(0., 4.),
}

priors = {**cosmo_rate_priors, **mass_priors, **scale_priors, **w_out_priors}
params_keys = list(priors.keys())
n_dim = len(params_keys)
prior_dists = [priors[k] for k in params_keys]

def prior_transform(u):
  return np.array([dist.ppf(u[i]) for i, dist in enumerate(prior_dists)])

print(f"n_dim = {n_dim}  ({n_features} w_out + {n_scales} feature_scales + {len(mass_priors)} mass + {len(cosmo_rate_priors)} cosmo/rate)")

## Section 0 -- local sanity checks (no cluster data / `tempest` needed)

### 0.1 Prior-predictive draws across different lengthscale blocks

Draw `feature_scales` and `w_out` straight from the priors above, build a
`random_features_density` realization for each draw, and overlay `p(m1)`.
Because the 4 scale priors are narrow, each draw is a small perturbation of
the default "blocks" -- this checks that the model behaves smoothly as
`feature_scales` moves around its prior, not that it produces wildly
different basis structures.

In [ ]:
n_draws = 12
m_low_fixed, m_high_fixed = 0.92, 124.2  # fixed mass range for this prior-predictive check

base_model = random_features_density(
  n_features=n_features, feature_scales=default_feature_scales,
  m_low=m_low_fixed, m_high=m_high_fixed, key=master_key,
)

rng = np.random.default_rng(1)
m1 = jnp.linspace(m_low_fixed * 1.01, m_high_fixed * 0.99, 4000)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
drawn_scales = []
for i in range(n_draws):
  fs = jnp.array([scale_priors[f"scale{j+1}"].rvs(random_state=rng) for j in range(n_scales)])
  w_out = jnp.array([w_out_priors[f"w_out{j}"].rvs(random_state=rng) for j in range(n_features)])
  drawn_scales.append(np.asarray(fs))

  model_i = base_model.update(feature_scales=fs, w_out=w_out)
  ax.plot(m1, p_m1(model_i, m1), alpha=0.7, lw=1.2)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$')
ax.set_title(f'{n_draws} prior-predictive draws of the 4 lengthscale blocks')
fig.tight_layout()
plt.show()

drawn_scales = np.array(drawn_scales)
print("drawn feature_scales -- min/max per block:")
for j in range(n_scales):
  print(f"  block {j+1} (default {default_feature_scales[j]:.3f}): "
        f"[{drawn_scales[:, j].min():.4f}, {drawn_scales[:, j].max():.4f}]")

### 0.2 Are the 4 blocks' priors actually narrow and non-overlapping?

Histogram many draws per block against the prior bounds.

In [ ]:
n_hist = 4000
fig, axes = plt.subplots(1, n_scales, figsize=(4 * n_scales, 3.2), sharey=False)
for j, ax in enumerate(axes):
  draws = scale_priors[f"scale{j+1}"].rvs(size=n_hist, random_state=rng)
  ax.hist(draws, bins=40, color=f"C{j}", alpha=0.8)
  ax.axvline(default_feature_scales[j], color='k', ls='--', lw=1, label='default')
  ax.set_title(f"block {j+1} prior")
  ax.set_xlabel("feature_scales[%d]" % j)
  ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 0.3 jit/vmap check: one compile, many `feature_scales` values

This is the actual point of the `nn.py` fix: batch several particles with
*different* `feature_scales` and `w_out` through `.update()` + `p_m1` under a
single `jax.jit` + `jax.vmap`, exactly the way `tempest` will evaluate a
particle population. The second call reuses the trace (no recompilation)
even though its `feature_scales` values differ from the first call's.

In [ ]:
import time

n_particles_test = 64
fs_batch = jnp.stack([
  jnp.array([scale_priors[f"scale{j+1}"].rvs(random_state=rng) for j in range(n_scales)])
  for _ in range(n_particles_test)
])
w_out_batch = jnp.stack([
  jnp.array([w_out_priors[f"w_out{j}"].rvs(random_state=rng) for j in range(n_features)])
  for _ in range(n_particles_test)
])

@jax.jit
def eval_particle(fs, w_out):
  model_i = base_model.update(feature_scales=fs, w_out=w_out)
  return p_m1(model_i, m1)

t0 = time.time()
out1 = jax.vmap(eval_particle)(fs_batch, w_out_batch)
out1.block_until_ready()
t1 = time.time()

fs_batch_2 = fs_batch * 1.05  # different concrete values, same shapes
out2 = jax.vmap(eval_particle)(fs_batch_2, w_out_batch)
out2.block_until_ready()
t2 = time.time()

print(f"first call (compile + run): {t1 - t0:.3f}s")
print(f"second call, different feature_scales values (should be much faster -- no retrace): {t2 - t1:.3f}s")
assert not jnp.any(jnp.isnan(out1)) and not jnp.any(jnp.isnan(out2))
print("no NaNs -- vmap over distinct feature_scales works with a single jit trace.")

## Section 1 -- full `tempest` hyperlikelihood run (cluster only)

Mirrors `examples/bpl3p_run_tempest.py` exactly in structure: load GWTC5 PE
+ injection data, build `cosmo`/`mass`/`rate` -> `population` ->
`hyperlikelihood`, define `log_likelihood`, run `tempest.Sampler`, save
posterior + blobs to `arviz.InferenceData`.

Needs `dir_data`/`dir_chain` (set in the config cell above) to point at real
GWTC5 files, and the `tempest` package (both cluster-only) -- **edit the
paths before running this section**.

In [ ]:
from CHIMERA import data
from CHIMERA.cosmo import flrw
from CHIMERA.rate import madau_dickinson
from CHIMERA import population as population_module, selection_function, hyperlikelihood
from CHIMERA.utils import emcee_utils as eu
import h5py
import tempest as tp

## Load PE data
file_ev = dir_data + 'GWTC5_242CBC_FAR0.25_PE2048.h5'
pe_gw = data.load_gw_pe_samples(file_ev, parameters=['m1det', 'm2det', 'dL'], return_struct=True)
pe_prior = jnp.load(dir_data + 'GWTC5_242CBC_FAR0.25_PE2048prior.npy')
pe_gw = pe_gw.update(pe_prior=pe_prior)
Nev = len(pe_gw.dL)

## Load injection data and define selection function
file_inj = dir_data + "GWTC5_injections_far0.25.h5"
f_inj = h5py.File(file_inj)
Ngen = float(f_inj.attrs['Ngen'])
Tobs = float(f_inj.attrs['Tobs'])
theta_inj_det = data.load_injection_data(file_inj, snr_cut=None, key_mapping={'snr': 'snr', 'log_pdraw': 'log_pdraw'}, frame='detector')

sel_fcn = selection_function(theta_inj_det, N_inj=Ngen)

In [ ]:
cosmo = flrw(H0=67.9, Om0=0.3065, z_max=5.)
mass = random_features_density(
  n_features=n_features, feature_scales=default_feature_scales,
  m_low=0.9, m_high=124., key=master_key,
)
rate = madau_dickinson()
population = population_module(cosmo, mass, rate, scale_free=True, Tobs=Tobs, R0=1)

hyperlike = hyperlikelihood(
  theta_gw_det=pe_gw,
  population=population,
  z_grids_res=300,
  selection_function=sel_fcn,
  pe_neff=2.0,
  inj_neff=None,
  kind_kde='fft',
  kernel='epan',
  kde_bw=None,
)

`eu.generate_dict` turns the flat `tempest` particle vector into a
dict keyed by `params_keys` -- but it has no special-cased handling for the
`w_out{i}`/`scale{i}` families the way it does for `spline_c{i}`, so we pop
those out and re-stack them into the `w_out`/`feature_scales` arrays that
`random_features_density.update()` (via `mass.keys`) expects.

In [ ]:
def log_likelihood(params):
  raw = eu.generate_dict(params, params_keys)
  w_out = jnp.array([raw.pop(f"w_out{i}") for i in range(n_features)])
  feature_scales = jnp.array([raw.pop(f"scale{i+1}") for i in range(n_scales)])
  hyperparams = {**raw, "w_out": w_out, "feature_scales": feature_scales}
  log_like_evs, N_exp, neff_inj, log_like = hyperlike.compute_all(**hyperparams)
  return log_like, log_like_evs, N_exp, neff_inj

# quick single-point smoke test before launching the full sampler
theta0 = prior_transform(np.random.default_rng(0).uniform(size=n_dim))
ll0, ll_evs0, N_exp0, neff_inj0 = log_likelihood(theta0)
print(f"log_likelihood at one prior draw: {ll0:.3f}  (N_exp={float(N_exp0):.2f}, neff_inj={float(neff_inj0):.2f})")

In [ ]:
sampler = tp.Sampler(
  prior_transform=prior_transform,
  log_likelihood=log_likelihood,
  n_dim=n_dim,
  n_particles=n_particles,
  n_steps=n_steps,
  resample=resample,
  random_state=prng_key,
  output_dir=dir_chain,
  output_label=chain_label,
)

sampler.run(n_total=n_total, save_every=1, resume_state_path=None)

samples, weights, logl, blobs = sampler.posterior(resample=True, return_blobs=True)
samples_dict = {k: samples[:, i] for i, k in enumerate(priors)}

log_like_evs_blob = blobs[:, 0]
N_exp_blob = blobs[:, 1]
neff_inj_blob = blobs[:, 2]

logZ, logZerr = sampler.evidence()
print(f"logZ = {logZ}  logZerr = {logZerr}")

In [ ]:
import xarray as xr
import arviz as az

n_chains = 1
n_draws_post = len(samples)
n_events = np.stack(log_like_evs_blob).shape[-1]

posterior_ds = xr.Dataset(
  {k: (['chain', 'draw'], v.reshape(n_chains, n_draws_post)) for k, v in samples_dict.items()}
)
sample_stats_ds = xr.Dataset({
  'log_likelihood': (['chain', 'draw'], logl.reshape(n_chains, n_draws_post)),
  'weights': (['chain', 'draw'], weights.reshape(n_chains, n_draws_post)),
  'log_like_evs': (['chain', 'draw', 'event'], np.stack(log_like_evs_blob).reshape(n_chains, n_draws_post, n_events)),
  'N_exp': (['chain', 'draw'], np.asarray(N_exp_blob).reshape(n_chains, n_draws_post)),
  'Neff_inj': (['chain', 'draw'], np.asarray(neff_inj_blob).reshape(n_chains, n_draws_post)),
})

idata = az.InferenceData(posterior=posterior_ds, sample_stats=sample_stats_ds)
idata.attrs['log_evidence'] = float(logZ)
idata.attrs['log_evidence_err'] = str(logZerr) if logZerr is None else float(logZerr)
idata.to_netcdf(f"{dir_chain}/{chain_label}_final.nc")

## Section 2 -- inspecting the 4 blocks in the posterior

Compares the posterior over the 4 `feature_scales` against their (narrow)
priors -- if the posterior tightens noticeably relative to the prior for a
block, the data is informative about that lengthscale; if it just traces the
prior, that block isn't constrained by this dataset/`n_features` and could
be dropped or widened. Also draws a posterior-predictive `p(m1)` band using
the sampled `w_out`/`feature_scales`/mass-edge hyperparameters together.

In [ ]:
fig, axes = plt.subplots(1, n_scales, figsize=(4 * n_scales, 3.2))
for j, ax in enumerate(axes):
  post = samples_dict[f"scale{j+1}"]
  prior_draws = scale_priors[f"scale{j+1}"].rvs(size=5000, random_state=np.random.default_rng(j))
  ax.hist(prior_draws, bins=40, density=True, alpha=0.4, label='prior', color='gray')
  ax.hist(post, bins=40, density=True, alpha=0.7, label='posterior', color=f"C{j}")
  ax.axvline(default_feature_scales[j], color='k', ls='--', lw=1)
  ax.set_title(f"block {j+1}")
  ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
n_ppc = 200
idx = np.random.default_rng(0).choice(len(samples), size=n_ppc, replace=False)
m1_grid = jnp.linspace(0.5, 150., 3000)

curves = []
for i in idx:
  w_out_i = jnp.array([samples_dict[f"w_out{k}"][i] for k in range(n_features)])
  fs_i = jnp.array([samples_dict[f"scale{k+1}"][i] for k in range(n_scales)])
  model_i = mass.update(
    w_out=w_out_i, feature_scales=fs_i,
    beta=samples_dict['beta'][i], m_low=samples_dict['m_low'][i], m_high=samples_dict['m_high'][i],
  )
  curves.append(p_m1(model_i, m1_grid))

curves = jnp.stack(curves)
lo, med, hi = jnp.percentile(curves, jnp.array([5., 50., 95.]), axis=0)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.fill_between(m1_grid, lo, hi, alpha=0.3, label='90% posterior-predictive')
ax.plot(m1_grid, med, lw=1.5, label='median')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$')
ax.legend()
fig.tight_layout()
plt.show()